# Day S3 — 치환 행렬: PAM과 BLOSUM은 어떻게 만들어졌나

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/supplement/day-S3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/supplement/day-S3.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — log-odds 점수를 손으로

In [ ]:
import math

def log_odds(q_pair, p_a, p_b):
    """관찰된 쌍 빈도 / 우연히 기대되는 빈도 → 2 × log2 점수(half-bit 단위)."""
    if p_a == p_b:
        expected = p_a * p_b            # 같은 글자끼리: p × p
    else:
        expected = 2 * p_a * p_b        # 다른 글자끼리: 순서가 두 가지(a-b, b-a)
    ratio = q_pair / expected
    return ratio, 2 * math.log2(ratio)

cases = [
    ("L–L (흔하고 잘 보존)", 0.040, 0.10, 0.10),
    ("W–W (드물고 잘 보존)", 0.006, 0.01, 0.01),
    ("L–I (비슷한 성질)  ", 0.018, 0.10, 0.06),
    ("L–D (성질이 다름)  ", 0.002, 0.10, 0.05),
]
for name, q, pa, pb in cases:
    ratio, score = log_odds(q, pa, pb)
    print(f"{name}  관찰/기대 = {ratio:5.2f}배  점수 = {score:+.1f} → {round(score):+d}")

### 예시 2 — 작은 블록으로 미니 BLOSUM 만들기

In [ ]:
import math
from itertools import combinations
import pandas as pd

block = [            # 가상의 보존 구간: 서열 8개 × 8칸, gap 없음
    "KLDEKLLD",
    "KLDERLLD",
    "RLEEKILD",
    "RIDDKLLE",
    "KLDEKLID",
    "RLDERILD",
    "KIEEKLLD",
    "KLDDRLLE",
]
letters = sorted(set("".join(block)))

pair_count = {}                              # 같은 칸(column) 안에서 만들 수 있는 모든 쌍을 센다
for col in range(len(block[0])):
    column = [seq[col] for seq in block]
    for a, b in combinations(column, 2):
        key = tuple(sorted((a, b)))
        pair_count[key] = pair_count.get(key, 0) + 1

total_pairs = sum(pair_count.values())
q = {key: n / total_pairs for key, n in pair_count.items()}     # 관찰된 쌍 빈도

p = {}                                                           # 각 글자의 배경 빈도
for x in letters:
    p[x] = q.get((x, x), 0) + sum(v for k, v in q.items() if x in k and k[0] != k[1]) / 2
assert abs(sum(p.values()) - 1) < 1e-9

score = pd.DataFrame("", index=letters, columns=letters)
for a in letters:
    for b in letters:
        key = tuple(sorted((a, b)))
        if key not in q:
            score.loc[a, b] = "."                                # 한 번도 관찰되지 않은 쌍
            continue
        expected = p[a] * p[b] if a == b else 2 * p[a] * p[b]
        score.loc[a, b] = f"{round(2 * math.log2(q[key] / expected)):+d}"

print("전체 쌍 개수:", total_pairs)
print("배경 빈도:", {x: round(v, 3) for x, v in p.items()})
print(score)

### 예시 3 — PAM의 발상: 행렬을 거듭 곱하기

In [ ]:
import numpy as np
import pandas as pd

letters = ["A", "S", "W"]                 # 설명용 3글자 알파벳
f = np.array([0.5, 0.4, 0.1])             # 배경 빈도

# '1% 변화' 행렬: M[i, j] = 글자 i가 한 단위 시간 뒤 j로 바뀌어 있을 확률 (가상 값)
M1 = np.array([
    [0.9900, 0.0096, 0.0004],
    [0.0120, 0.9875, 0.0005],
    [0.0020, 0.0020, 0.9960],
])
assert np.allclose(M1.sum(axis=1), 1)
print("한 단위에서 바뀌는 비율:", round(float(1 - (f * np.diag(M1)).sum()), 4))

for n in [1, 50, 250, 2000]:
    Mn = np.linalg.matrix_power(M1, n)          # 같은 행렬을 n번 곱한다
    unchanged = (f * np.diag(Mn)).sum()
    S = 10 * np.log10(Mn / f)                   # 열 j의 배경 빈도로 나눈 뒤 로그
    print(f"\nPAM{n}: 그대로 남은 자리 {unchanged:.0%}")
    print(pd.DataFrame(S.round().astype(int), index=letters, columns=letters))